# Issue 304: real TinyLlama shadow-TPE benchmark

This notebook contains **no measured results**. All objectives below require actual CUDA
LoRA training. The synthetic benchmark is unrelated evidence.

Run from the repository root in a dedicated, already configured RapidFire **fit** environment
(Python 3.12, CUDA/BF16 GPU, Transformers, TRL, PEFT, Optuna, datasets, evaluate,
rouge-score, BLEU dependencies and MLflow). Nothing here installs dependencies.
Use a local Jupyter kernel, not Colab's background execution path. Do not run other
experiments against the same RapidFire database. Reserve the same GPU set for every mode.

1. Select preparation below and run through the artifact cell once, online. This writes
   an immutable lock for the current environment and caches pinned artifacts. Archive that
   lock and `pip-freeze.txt`; recreate the same environment for replications.
2. Restart the kernel, disable preparation, enable GPU execution, and run all cells.
   Run the forced-pruning pilot first. Its scores are **excluded** from comparisons.
3. Run all modes for seeds 42, 43, 44; rotate mode order by seed. Archive the output directory,
   lock, notebook, git diff and logs. A failed audit invalidates the run.

The search template and metric/model functions are copied from the reference
`rf-tutorial-optuna-sft-chatqa-tiny.ipynb`. That tutorial is not modified.


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata as metadata, json, math, os, platform
import subprocess, time, uuid

ROOT = Path.cwd()
assert (ROOT / "rapidfireai/fit/backend/controller.py").exists(), "Start at repository root"
PREPARE = False               # Set True only to create a new lock/cache; then restart.
RUN_GPU = False               # Explicit maintainer opt-in; never enabled by CPU checks.
OUT = ROOT / "benchmarks/304_results"
LOCK_PATH = OUT / "lock.json"
SEEDS = [42, 43, 44]
MODES = ["rf_default", "rf_shadow", "sequential"]
N_INITIAL, BUDGET, NUM_SHARDS, MAX_STEPS, EVAL_STEPS = 4, 8, 4, 128, 4
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"


def dump(path, value):
    path.write_text(json.dumps(value, indent=2, default=str, allow_nan=False) + "\n")


def source_hashes():
    paths = subprocess.check_output(
        ["git", "ls-files", "rapidfireai", "pyproject.toml", "requirements.txt"], text=True
    ).splitlines()
    return {p: hashlib.sha256((ROOT / p).read_bytes()).hexdigest() for p in paths}


def environment():
    return {d.metadata["Name"].lower(): d.version for d in metadata.distributions()}


## Freeze and cache before measuring

Preparation resolves Hub refs **once**, stores their immutable commit SHAs, pins every
installed distribution and hashes repository sources (including the shard-attribution fix).
Measured runs verify this lock; they do not silently refresh it. This is a captured
reproducibility profile, not a claim that an untested dependency combination is supported.
Use a new output directory to create a different profile. Metrics use the pinned Evaluate
ROUGE/BLEU implementations; generation arguments and four-decimal rounding match the tutorial.


In [ ]:
if PREPARE:
    assert not LOCK_PATH.exists(), "Refusing to overwrite the reproducibility lock"
    from huggingface_hub import HfApi, snapshot_download
    import evaluate
    api = HfApi()
    lock = {
        "model": api.model_info(MODEL).sha,
        "dataset": api.dataset_info(DATASET).sha,
        "rouge": api.space_info("evaluate-metric/rouge").sha,
        "bleu": api.space_info("evaluate-metric/bleu").sha,
        "packages": environment(), "sources": source_hashes(),
        "python": platform.python_version(),
        "git_head": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    }
    snapshot_download(MODEL, revision=lock["model"])
    from datasets import load_dataset
    load_dataset(DATASET, revision=lock["dataset"])
    for metric in ["rouge", "bleu"]:
        evaluate.load(metric, revision=lock[metric])
    OUT.mkdir(parents=True, exist_ok=True)
    dump(LOCK_PATH, lock)
    (OUT / "pip-freeze.txt").write_text(subprocess.check_output(
        [os.sys.executable, "-m", "pip", "freeze", "--all"], text=True))
    print("Prepared. Restart, set PREPARE=False and RUN_GPU=True.")

if RUN_GPU:
    assert not PREPARE
    LOCK = json.loads(LOCK_PATH.read_text())
    assert LOCK["packages"] == environment(), "Package versions changed"
    assert LOCK["sources"] == source_hashes(), "Repository sources changed"
    assert LOCK["python"] == platform.python_version()
    os.environ["RF_MLFLOW_ENABLED"] = "true"
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    os.environ["HF_EVALUATE_OFFLINE"] = "1"
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


In [ ]:
def sample_formatting_function(row):
    return {
        "prompt": [
            {"role": "system", "content": "You are a helpful and friendly customer support assistant. Please answer the user's query to the best of your ability."},
            {"role": "user", "content": row["instruction"]},
        ],
        "completion": [{"role": "assistant", "content": row["response"]}],
    }


def sample_compute_metrics(eval_preds):
    import evaluate
    predictions, labels = eval_preds
    # Keep tutorial semantics, including Evaluate's aggregation and rounding.
    rouge = evaluate.load("rouge", revision=LOCK["rouge"])
    bleu = evaluate.load("bleu", revision=LOCK["bleu"])
    return {
        "rougeL": round(rouge.compute(predictions=predictions, references=labels,
                                      use_stemmer=True)["rougeL"], 4),
        "bleu": round(bleu.compute(predictions=predictions, references=labels)["bleu"], 4),
    }


def sample_create_model(model_config):
    from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
    set_seed(model_config["training_args"]["seed"])
    model = AutoModelForCausalLM.from_pretrained(
        model_config["model_name"], **model_config["model_kwargs"])
    tokenizer = AutoTokenizer.from_pretrained(model_config["model_name"], revision=LOCK["model"])
    # Each call loads the pinned base, never a previous trial's adapter.
    return model, tokenizer


def template(seed, max_steps=MAX_STEPS, sampling_only=False):
    from rapidfireai.automl import List, RFModelConfig
    if sampling_only:
        # Parameter containers only: exercise real RFOptuna without PEFT/TRL or CUDA.
        # RF's search extractor and leaf expansion consume these _user_params.
        from types import SimpleNamespace
        def parameter_container(**kwargs):
            return SimpleNamespace(_user_params=kwargs)
        RFLoraConfig = RFSFTConfig = parameter_container
    else:
        from rapidfireai.automl import RFLoraConfig, RFSFTConfig
    model_revision = "sampling-only" if sampling_only else LOCK["model"]
    return RFModelConfig(
        model_name=MODEL,
        peft_config=RFLoraConfig(
            r=List([8, 32]), lora_alpha=List([16, 32, 64, 128]), lora_dropout=0.1,
            target_modules=List([["q_proj", "v_proj"],
                                 ["q_proj", "k_proj", "v_proj", "o_proj"]]), bias="none"),
        training_args=RFSFTConfig(
            learning_rate=List([1e-3, 1e-4]), lr_scheduler_type="linear",
            per_device_train_batch_size=4, per_device_eval_batch_size=4,
            max_steps=max_steps, gradient_accumulation_steps=1,
            logging_steps=2, eval_strategy="steps", eval_steps=EVAL_STEPS,
            bf16=True, seed=seed, data_seed=seed),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False,
                      "revision": model_revision},
        formatting_func=sample_formatting_function, compute_metrics=sample_compute_metrics,
        generation_config={"max_new_tokens": 256, "temperature": 0.8, "top_p": 0.9,
                           "top_k": 30, "repetition_penalty": 1.05})


def validate_fixed_shards(config, train_size=128):
    from rapidfireai.fit.backend.shards import DatasetShards
    args = config.training_args._user_params
    assert train_size == 128 and NUM_SHARDS == 4
    assert args["per_device_train_batch_size"] == 4
    assert args["gradient_accumulation_steps"] == 1
    assert config.num_gpus in (None, 1), "Each trial must use one worker"
    assert not args.get("fsdp_config") and not args.get("packing", False)
    assert args["eval_strategy"] == "steps" and args["eval_steps"] == EVAL_STEPS == 4
    assert args["max_steps"] in (16, MAX_STEPS) and MAX_STEPS == 128
    assert args["max_steps"] % 8 == 0
    shards = DatasetShards(train_size, NUM_SHARDS, batch_size=4)
    lengths = [end - start for start, end in
               (shards.get_shard_indices(i) for i in range(NUM_SHARDS))]
    assert lengths == [32] * NUM_SHARDS
    assert all(math.ceil(length / 4) == 8 for length in lengths)


def preflight_initial_proposals():
    from rapidfireai.automl import RFOptuna
    import random
    # get_runs seeds Python's RNG; restore it so this check cannot alter scheduling.
    state = random.getstate()
    proposals = {}
    try:
        for seed in SEEDS:
            paired = {}
            for mode in ("rf_default", "rf_shadow"):
                config = template(seed, sampling_only=True)
                validate_fixed_shards(config)
                validate_fixed_shards(template(seed, 16, sampling_only=True))
                group = RFOptuna([config], trainer_type="SFT", n_initial=N_INITIAL,
                    budget=BUDGET, objective="maximize:rougeL", sampler="tpe",
                    pruner="median", seed=seed, interim_tpe=(mode == "rf_shadow"))
                group.get_runs(seed)
                paired[mode] = [dict(trial.params) for trial in group._study.trials]
                assert len(paired[mode]) == N_INITIAL
            assert paired["rf_default"] == paired["rf_shadow"], (seed, paired)
            proposals[seed] = paired["rf_default"]
    finally:
        random.setstate(state)
    return proposals


# Runs without a model load, dataset download, PEFT/TRL, or GPU execution.
initial_proposal_preflight = preflight_initial_proposals()
print("CPU preflight passed: eight-step shards and paired initial proposals", SEEDS)


## Audit at the real shard callback

128 training examples / batch 4 / four shards = eight optimizer steps per shard.
Evaluation must produce fresh ROUGE-L **and BLEU** at every multiple of four, including
the last step. The controller skips the callback at the final training boundary, so
`finalize` audits that boundary separately. Missing/OOM-suppressed metrics invalidate a run.
Audit errors are retained because the controller catches callback exceptions.
The CPU preflight above checks the fixed shard configuration and compares real RFOptuna
initial proposals for every seed using parameter-only containers; it never loads a model.
The same configuration checks run on real training configs before GPU execution.

The notebook uses existing private RFOptuna helpers solely to reuse its exact parameter
names, categorical encoding and config-leaf expansion for sequential sampling. Source
hashes guard these interfaces. No library monkey-patches or sampler changes are made.


In [ ]:
def check_history(metrics, boundary):
    for key in ("rougeL", "bleu"):
        history = metrics.get(key, [])
        observed = {int(step): float(value) for step, value in history}
        expected = set(range(EVAL_STEPS, boundary + 1, EVAL_STEPS))
        assert set(observed) == expected, (key, boundary, sorted(observed))
        assert all(math.isfinite(v) for v in observed.values()), key
    return dict(metrics["rougeL"])[boundary]


def make_adapters():
    import optuna
    from rapidfireai.automl import RFOptuna
    from rapidfireai.automl.callbacks import RunDecision
    from rapidfireai.automl.optuna_search import (
        _extract_search_space, _sample_from_trial_multi, _template_to_leaf_fit)
    from rapidfireai.fit.db.rf_db import RfDb

    class Audit:
        def __init__(self, delegate, study, sequential=False):
            self.delegate, self.study, self.sequential = delegate, study, sequential
            self.events, self.errors, self.mapping, self.steps = [], [], {}, {}

        def __getattr__(self, name):
            return getattr(self.delegate, name)

        def _set_initial_trials(self, trial_map, spawned):
            self.delegate._set_initial_trials(trial_map, spawned)
            self.mapping.update({rid: trial.number for rid, trial in trial_map.items()})

        def _remap_pending_trial(self, run_id):
            self.delegate._remap_pending_trial(run_id)
            self.mapping[run_id] = self.delegate._trials[run_id].number

        def on_shard_complete(self, run_id, shard_id, metrics):
            try:
                trial = self.delegate._trials[run_id]
                self.mapping[run_id] = trial.number
                step = self.steps.get(run_id, 0)
                boundary = (step + 1) * 8
                row = RfDb().get_run(run_id)
                assert row["completed_steps"] == boundary
                assert shard_id == step % NUM_SHARDS
                value = check_history(metrics, boundary)
                if step == 0:
                    assert row["warm_started_from"] is None and row["cloned_from"] is None
                    assert row["shard_offset"] == 0
                    # Replay the real trial's encoded params without asking the study.
                    expected = _template_to_leaf_fit(_sample_from_trial_multi(
                        optuna.trial.FixedTrial(trial.params),
                        self.delegate._config_templates, self.delegate._search_spaces,
                        range_cache=self.delegate._range_cache), "SFT")
                    actual = row["config_leaf"]
                    assert actual["peft_params"] == expected["peft_params"]
                    assert actual["training_args"]["learning_rate"] == expected["training_args"]["learning_rate"]
                    assert actual["model_name"] == MODEL
                self.steps[run_id] = step + 1
                if self.sequential:
                    # Explicit report/prune; final tell occurs in finalize below.
                    trial.report(value, step)
                    if trial.should_prune():
                        self.study.tell(trial, state=optuna.trial.TrialState.PRUNED)
                        decision = RunDecision(action="prune")
                    else:
                        decision = RunDecision(action="continue")
                else:
                    decision = self.delegate.on_shard_complete(run_id, shard_id, metrics)
                self.events.append(dict(t=time.time(), run_id=run_id, trial=trial.number,
                    shard_id=shard_id, step=step, optimizer_step=boundary, rougeL=value,
                    action=decision.action, replacement=decision.replacement_config is not None))
                return decision
            except Exception as exc:
                self.errors.append(repr(exc))
                raise

        def finalize(self, final_metrics):
            try:
                for rid, trial in self.delegate._trials.items():
                    assert isinstance(rid, int), "Unmapped replacement trial"
                    row = RfDb().get_run(rid)
                    metrics = final_metrics.get(rid, {})
                    frozen = self.study.trials[trial.number]
                    if frozen.state == optuna.trial.TrialState.RUNNING:
                        assert row["completed_steps"] == row["total_steps"]
                        check_history(metrics, row["total_steps"])
                    if self.sequential and frozen.state == optuna.trial.TrialState.RUNNING:
                        self.study.tell(trial, check_history(metrics, row["total_steps"]))
                if not self.sequential:
                    self.delegate.finalize(final_metrics)
            except Exception as exc:
                self.errors.append(repr(exc))
                raise

    class AuditedRF(RFOptuna):
        def get_callback(self, num_shards=None):
            if self._study is None:
                return None
            delegate = super().get_callback(num_shards=num_shards)
            self._callback = Audit(delegate, self._study)
            return self._callback

    class OneTrial(AuditedRF):
        def __init__(self, study, trial, config, seed):
            super().__init__([config], trainer_type="SFT", n_initial=1, budget=1,
                             objective="maximize:rougeL", seed=seed)
            self.external_study, self.external_trial = study, trial

        def get_runs(self, seed=42):
            self._study = self.external_study
            self._initial_trials = [self.external_trial]
            self._config_templates = list(self.configs)
            self._search_spaces = [_extract_search_space(c) for c in self.configs]
            sampled = _sample_from_trial_multi(
                self.external_trial, self._config_templates, self._search_spaces,
                range_cache=self._range_value_cache)
            return [_template_to_leaf_fit(sampled, "SFT")]

        def get_callback(self, num_shards=None):
            callback = super().get_callback(num_shards)
            if callback is not None:
                callback.sequential = True
            return callback

    return AuditedRF, OneTrial


## Runtime and GPU telemetry

`nvidia-smi` samples every physical GPU once per second: UUID, utilization, memory and
power. Use a dedicated GPU host: nvidia-smi enumerates physical devices regardless of
CUDA_VISIBLE_DEVICES. These are device-wide observations, not per-trial attribution. Raw samples are exported. Runtime includes model creation, scheduling,
checkpointing and evaluation; artifact downloads were excluded during preparation.
Sequential trials include repeated experiment/worker startup, an explicit overhead difference.


In [ ]:
import threading

class GPUTrace:
    def __init__(self):
        self.rows, self.errors, self.stop = [], [], threading.Event()

    def sample(self):
        while not self.stop.is_set():
            try:
                raw = subprocess.check_output([
                    "nvidia-smi", "--query-gpu=uuid,utilization.gpu,memory.used,power.draw",
                    "--format=csv,noheader,nounits"], text=True)
                for line in raw.splitlines():
                    gpu, util, mem, power = [s.strip() for s in line.split(",")]
                    self.rows.append(dict(t=time.time(), gpu=gpu, utilization=float(util),
                                          memory_mib=float(mem), power_w=float(power)))
            except Exception as exc:
                self.errors.append(repr(exc))
            self.stop.wait(1)

    def __enter__(self):
        self.thread = threading.Thread(target=self.sample, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, *args):
        self.stop.set()
        self.thread.join()


def gpu_summary(rows):
    result = {}
    for gpu in sorted({row["gpu"] for row in rows}):
        samples = sorted([r for r in rows if r["gpu"] == gpu], key=lambda r: r["t"])
        result[gpu] = dict(
            peak_memory_mib=max(r["memory_mib"] for r in samples),
            mean_utilization=sum(r["utilization"] for r in samples) / len(samples),
            sampled_energy_j=sum((b["t"]-a["t"])*(a["power_w"]+b["power_w"])/2
                                 for a, b in zip(samples, samples[1:])))
    return result


In [ ]:
def train_group(group, seed, label, train, validation, directory):
    from rapidfireai import Experiment
    from rapidfireai.fit.db.rf_db import RfDb
    from rapidfireai.utils.metric_rfmetric_manager import RFMetricLogger
    experiment = Experiment(experiment_name=f"b304-{label}-{uuid.uuid4().hex[:8]}", mode="fit")
    error = None
    try:
        experiment.run_fit(group, sample_create_model, train, validation,
                           num_shards=NUM_SHARDS, seed=seed, num_gpus=1)
        assert not getattr(experiment, "_training_thread", None), "Use a blocking local kernel"
    except Exception as exc:
        error = repr(exc)
    finally:
        callback = group._callback
        # Read before end() cleans up the experiment database.
        rows = RfDb().get_all_runs()
        ids = set(callback.mapping) if callback else set()
        selected = {rid: rows[rid] for rid in ids}
        metric_logger = RFMetricLogger(RFMetricLogger.get_default_metric_loggers(
            experiment_name=experiment.experiment_name))
        histories = {rid: metric_logger.get_run_metrics(row["metric_run_id"])
                     for rid, row in selected.items()}
        payload = dict(error=error, errors=callback.errors if callback else ["No callback"],
            events=callback.events if callback else [], mapping=callback.mapping if callback else {},
            runs=selected, histories=histories)
        dump(directory / f"{label}-audit.json", payload)
        experiment.end()
    assert error is None, error
    assert callback and not callback.errors, payload["errors"]
    assert len(callback.mapping) == len(callback.delegate._trials)
    assert len(set(callback.mapping.values())) == len(callback.mapping)
    assert len(callback.mapping) == (1 if callback.sequential else len(group._study.trials))
    for rid, number in callback.mapping.items():
        state = group._study.trials[number].state.name
        assert selected[rid]["status"].name == {
            "COMPLETE": "COMPLETED", "PRUNED": "STOPPED", "FAIL": "FAILED"}[state]
    assert all(t.state.name in {"COMPLETE", "PRUNED", "FAIL"} for t in group._study.trials)
    return payload


def run_mode(mode, seed, pilot=False):
    import optuna, torch
    from datasets import load_dataset
    from transformers import set_seed
    validate_fixed_shards(template(seed, 16 if pilot else MAX_STEPS))
    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    AuditedRF, OneTrial = make_adapters()
    set_seed(seed)
    dataset = load_dataset(DATASET, revision=LOCK["dataset"])["train"]
    train = dataset.select(range(128)).shuffle(seed=seed)
    validation = dataset.select(range(128, 152)).shuffle(seed=seed)
    validate_fixed_shards(template(seed, 16 if pilot else MAX_STEPS), len(train))
    directory = OUT / f"{'pilot' if pilot else mode}-{seed}-{uuid.uuid4().hex[:8]}"
    directory.mkdir(parents=True)
    dump(directory / "manifest.json", dict(mode=mode, seed=seed, pilot=pilot,
         lock=LOCK, train_fingerprint=train._fingerprint, eval_fingerprint=validation._fingerprint,
         cuda=torch.version.cuda, devices=[torch.cuda.get_device_name(i)
             for i in range(torch.cuda.device_count())],
         cuda_visible_devices=os.environ.get("CUDA_VISIBLE_DEVICES"),
         nvidia_smi=subprocess.check_output(["nvidia-smi"], text=True)))
    started = time.time()
    audits, failure = [], None
    study = None
    with GPUTrace() as gpu:
        try:
            if mode == "sequential":
                study = optuna.create_study(direction="maximize",
                    sampler=optuna.samplers.TPESampler(seed=seed),
                    pruner=optuna.pruners.MedianPruner(n_startup_trials=2))
                for _ in range(BUDGET):
                    trial = study.ask()  # Only one outstanding trial.
                    group = OneTrial(study, trial, template(seed), seed)
                    try:
                        audits.append(train_group(group, seed, f"trial-{trial.number}",
                                                  train, validation, directory))
                    except Exception:
                        if study.trials[trial.number].state.name == "RUNNING":
                            study.tell(trial, state=optuna.trial.TrialState.FAIL)
                        raise
                    assert study.trials[trial.number].state.name != "RUNNING"
            else:
                group = AuditedRF([template(seed, 16 if pilot else MAX_STEPS)],
                    trainer_type="SFT", n_initial=2 if pilot else N_INITIAL,
                    budget=3 if pilot else BUDGET, objective="maximize:rougeL",
                    sampler="tpe", pruner="median", seed=seed,
                    interim_tpe=(mode == "rf_shadow"))
                if pilot:
                    # Force an actual prune after a real metric, not a simulated objective.
                    class PilotPruner(optuna.pruners.BasePruner):
                        def prune(self, study, trial):
                            return trial.number == 0 and bool(trial.intermediate_values)
                    group._create_pruner = lambda: PilotPruner()
                try:
                    audits.append(train_group(group, seed, "concurrent", train, validation, directory))
                finally:
                    study = group._study
            if pilot:
                assert any(e["replacement"] for a in audits for e in a["events"])
                assert len(study.trials) == 3
                assert any(t.state.name == "PRUNED" for t in study.trials)
                assert any(t.state.name == "COMPLETE" for t in study.trials)
                assert not any(t.state.name == "FAIL" for t in study.trials)
        except Exception as exc:
            failure = repr(exc)
        finally:
            gpu.stop.set()
            gpu.thread.join()
            trials = [] if study is None else [dict(number=t.number, state=t.state.name,
                params=t.params, value=t.value, intermediate_values=t.intermediate_values,
                datetime_start=t.datetime_start, datetime_complete=t.datetime_complete)
                for t in study.trials]
            dump(directory / "trials.json", trials)
            dump(directory / "gpu.json", gpu.rows)
            complete = [t["value"] for t in trials if t["state"] == "COMPLETE"]
            summary = dict(mode=mode, seed=seed, pilot=pilot, failure=failure,
                wall_seconds=time.time()-started, gpu_errors=gpu.errors,
                gpu=gpu_summary(gpu.rows), spawned=len(trials),
                states={s: sum(t["state"] == s for t in trials)
                        for s in ["COMPLETE", "PRUNED", "FAIL", "RUNNING", "WAITING"]},
                replacements=sum(e["replacement"] for a in audits for e in a["events"]),
                best_completed_rougeL=max(complete) if complete else None,
                valid=failure is None and bool(gpu.rows) and not gpu.errors)
            dump(directory / "summary.json", summary)
    assert summary["valid"], summary
    return summary


## Execute pilot, then paired comparisons

The pilot trains real models for 16 steps, forces trial 0 to prune at the first
8-step boundary, and requires a fresh replacement and at least one completed trial.
First-boundary audits require no clone/warm-start parent, zero shard offset, metric
history beginning at step 4, and a run configuration matching its Optuna trial parameters. These validate lifecycle initialization; they do not hash
optimizer tensors. Inspect the exported run configs and checkpoint logs if stronger
initialization evidence is required. The normal benchmark always uses median pruning.


In [ ]:
if RUN_GPU:
    # Do not discard previous progress accidentally when rerunning this cell.
    assert not (OUT / "comparison.json").exists(), "Archive the previous comparison first"
    results = []

    def record_run(mode, seed, pilot=False):
        try:
            result = run_mode(mode, seed, pilot=pilot)
        except BaseException as exc:
            # Also retain failures before run_mode creates its own artifact directory.
            failure = dict(mode=mode, seed=seed, pilot=pilot, valid=False,
                           failure=repr(exc), exception_type=type(exc).__name__, t=time.time())
            dump(OUT / "failure.json", failure)
            if not pilot:
                results.append(failure)
            dump(OUT / "comparison.json", results)
            raise
        if not pilot:
            results.append(result)
            dump(OUT / "comparison.json", results)
        return result

    pilot_result = record_run("rf_shadow", SEEDS[0], pilot=True)
    for index, seed in enumerate(SEEDS):
        order = MODES[index % len(MODES):] + MODES[:index % len(MODES)]
        for mode in order:
            record_run(mode, seed)
        # Confirm paired RF arms actually received identical initial proposals.
        paired = {}
        for mode in ("rf_default", "rf_shadow"):
            latest = max(OUT.glob(f"{mode}-{seed}-*/trials.json"), key=lambda p: p.stat().st_mtime)
            paired[mode] = [t["params"] for t in json.loads(latest.read_text())[:N_INITIAL]]
        try:
            assert paired["rf_default"] == paired["rf_shadow"], "Initial proposal mismatch"
        except AssertionError as exc:
            dump(OUT / "failure.json", dict(phase="post-training-proposal-audit",
                 seed=seed, valid=False, failure=repr(exc), t=time.time()))
            raise
    dump(OUT / "comparison.json", results)
    print(json.dumps(results, indent=2))
else:
    print("GPU execution disabled. No training results have been produced.")


## Interpret the exports fairly

- Both RF arms use four initial trials and an eight-trial **ceiling**, the same seeds,
  model, LoRA choices, data, 128-step limit, evaluation cadence and visible devices.
  Default and shadow initial proposals match. Replacements happen only after pruning;
  RF can finish below eight trials. Shadow studies use zero startup trials and exact-step
  interim evidence; default TPE keeps its ordinary startup behavior. Do not tune that away.
- Sequential Optuna explicitly asks one trial, trains through the same RF shard backend,
  reports at steps 8, 16, ... 120, and tells before asking again. `num_shards=1` is never
  used to imply sequential optimization. It performs eight trials, including replacements
  after completed trials; this differs from RF's prune-only refill policy.
- Sequential median pruning uses completed reference trials and two startup trials;
  RF additionally compares concurrent peers. This is a system comparison, not an isolated
  sampler-quality ablation. All arms evaluate at steps 4, 8, ... 128 and optimize the last
  ROUGE-L of completed trials. PRUNED values are not completed objectives.
- “Concurrent” describes outstanding RF trials and shard scheduling. Actual overlapping
  kernels depend on GPU count; a single GPU time-slices trials. Keep device allocations
  identical and report their count. `device_map="auto"` follows the tutorial; verify worker
  GPU visibility in the pilot before using multiple devices. BF16 support is required.
- Compare best **completed** ROUGE-L, state counts, actual optimizer steps from audits,
  elapsed time and sampled energy together. Derive incumbent-over-time curves from trial
  completion timestamps. For equal-time curves, censor to a shared duration; do not compare
  unequal full-run budgets as if equal compute. These trial-ceiling runs do not enforce a
  hard wall-time budget. Report startup overhead and repeat variability, not just one seed.
- Generation retains the tutorial's model-default `do_sample` behavior, temperature/top-p/
  top-k, answer-only decoding, stemming and rounding. Pinned model generation defaults
  matter. Seeded GPU kernels and scheduler completion order can still be nondeterministic.
  24 evaluation examples give a noisy objective; this benchmark does not establish general
  model quality. A separate holdout study would be needed for that claim.
- `summary.json`, `trials.json`, per-experiment `*-audit.json`, `gpu.json`, manifests and
  `comparison.json` are machine-readable. Audit files include complete metric histories,
  trial/run mapping, prune decisions, replacements and final run records. Failures invalidate
  a run; retain their artifacts. GPU energy uses trapezoidal sampling and excludes time
  outside the telemetry window. No synthetic or forced-pilot score is GPU benchmark evidence.

Unverified until a maintainer runs the pilot: package compatibility, model fit in GPU memory,
metric cache resolution offline, checkpoint restoration cadence, GPU isolation and final
tracking flush. No GPU execution was performed when this notebook was authored.
